# Experiment 98 — Temporal Calibration-Safe Support Control

Experiment 97 improved 11/12 dataset–horizon cells, but support correction harmed Exchange at H=720. This experiment asks whether that failure can be detected **before official validation**.

For each dataset and horizon, the official training windows are divided chronologically into a fit prefix, an overlap-exclusion gap, and a calibration tail. PatchTST and ShrinkAdaptive are trained only from the fit loader (calibration is used for early stopping). Correction strength is then estimated from calibration predictions and frozen before official-validation evaluation.

Methods reported: Independent, NaturalSupport, CalGlobal, CrossFitSafe, CalChannelShrink, and ValOracle (diagnostic upper bound only). The test split is never constructed.

In [ ]:
# 0. Protocol and Forecast-JEPA runtime
from pathlib import Path
from types import SimpleNamespace,ModuleType
import os,sys,json,math,random,gc,copy,subprocess
if sys.version_info < (3,10): raise RuntimeError('Python 3.10 or newer required. Current: '+sys.version)
print('Python runtime:',sys.version); print('Executable:',sys.executable)
PROJECT_ROOT=Path(os.environ.get('FORECAST_PROJECT_ROOT','/data/code/forecast_jepa'))
EXP_NAME='98_temporal_calibration_safe_support_control'
RESULT_DIR=PROJECT_ROOT/'results'/EXP_NAME; CKPT_DIR=PROJECT_ROOT/'checkpoints'/EXP_NAME
RESULT_DIR.mkdir(parents=True,exist_ok=True); CKPT_DIR.mkdir(parents=True,exist_ok=True)
DATASETS=['ETTm1','Weather','ETTh2','Exchange']; HORIZONS=[96,336,720]; SEEDS=[9801,9802,9803]
METHODS=['Independent','NaturalSupport','CalGlobal','CrossFitSafe','CalChannelShrink','ValOracle']
LOOKBACK=512; LABEL_LEN=48; CHUNK=24; PATCH_LEN=16; STRIDE=8
D_MODEL=128; N_HEADS=4; D_FF=256; N_LAYERS=3; DROPOUT=.10; MAX_DELTA=.5; GATE_INIT_LOGIT=-2.0
BATCH_SIZE=64; EVAL_BATCH=128; TRAIN_EPOCHS=20; ADAPTER_EPOCHS=12; PATIENCE=5
LR=1e-4; ADAPTER_LR=3e-4; WD=1e-4; SHRINK_LAMBDA=.02; MAX_GATE_DEVIATION=.35
CAL_FRACTION=.12; MIN_CAL_WINDOWS=256; USE_BF16=True; FORCE_RETRAIN=False; N_BOOT=3000
HORIZON=HORIZONS[0]; N_SLOTS=HORIZON//CHUNK; DATA={}
print(EXP_NAME,'training units=',len(DATASETS)*len(HORIZONS)*len(SEEDS)*2); print('Results:',RESULT_DIR)


In [ ]:
# 1. Locate/import the official Time-Series-Library
def valid_root(p):
    p=Path(p).expanduser()
    return p.resolve() if (p/'models'/'PatchTST.py').is_file() else None
candidates=[Path(os.environ.get('TSLIB_ROOT','/data/Time-Series-Library_v2')),Path('/data/Time-Series-Library_v2'),Path('/data/Time-Series-Library'),Path('/data/code/Time-Series-Library'),PROJECT_ROOT/'Time-Series-Library',PROJECT_ROOT.parent/'Time-Series-Library']
TSLIB_ROOT=next((q for q in map(valid_root,candidates) if q is not None),None)
if TSLIB_ROOT is None:
    for parent in (Path('/data/code'),Path('/data'),PROJECT_ROOT.parent):
        if not parent.is_dir(): continue
        hits=list(parent.glob('*/models/PatchTST.py'))+list(parent.glob('*/*/models/PatchTST.py'))
        if hits: TSLIB_ROOT=hits[0].parent.parent.resolve(); break
if TSLIB_ROOT is None:
    dst=PROJECT_ROOT/'Time-Series-Library'; dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.exists() and not valid_root(dst): raise FileNotFoundError(f'Incomplete checkout at {dst}; set TSLIB_ROOT or rename/remove it.')
    print('Official checkout not found; cloning to',dst)
    subprocess.run(['git','clone','--depth','1','https://github.com/thuml/Time-Series-Library.git',str(dst)],check=True)
    TSLIB_ROOT=valid_root(dst)
if TSLIB_ROOT is None: raise FileNotFoundError('Time-Series-Library unavailable')
sys.path.insert(0,str(TSLIB_ROOT)); print('Time-Series-Library:',TSLIB_ROOT)

# Some TSLib installations import optional Reformer code even though PatchTST never uses it.
def reformer_stub(name):
    m=ModuleType(name)
    class ReformerLayer:
        def __init__(self,*a,**k): raise RuntimeError('Reformer stub invoked')
    m.ReformerLayer=ReformerLayer; sys.modules[name]=m
try:
    from models.PatchTST import Model as OfficialPatchTST
except ModuleNotFoundError as e:
    if e.name not in {'reformer','reformer_pytorch'}: raise
    reformer_stub(e.name); from models.PatchTST import Model as OfficialPatchTST
try:
    from data_provider.data_factory import data_provider
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(f'Missing dependency {e.name!r} while importing the official data provider. Install the repository requirements in the Forecast-JEPA kernel.') from e
print('Official PatchTST:',OfficialPatchTST)


In [ ]:
# 2. Leakage-controlled temporal data split; official validation remains untouched
import numpy as np,pandas as pd
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import DataLoader,Subset
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('CUDA required')
device=torch.device('cuda'); torch.set_float32_matmul_precision('high')
print(torch.__version__,torch.cuda.get_device_name(0),'BF16',torch.cuda.is_bf16_supported())
def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def locate(names):
    for root in [Path('/data/dataset'),Path('/data'),TSLIB_ROOT/'dataset']:
        if not root.exists(): continue
        for name in names:
            hits=list(root.glob('**/'+name))
            if hits: return sorted(hits,key=lambda p:len(str(p)))[0].resolve()
    raise FileNotFoundError('Could not locate '+str(names))
CSV={'ETTm1':locate(['ETTm1.csv']),'Weather':locate(['weather.csv']),'ETTh2':locate(['ETTh2.csv']),'Exchange':locate(['exchange_rate.csv','Exchange.csv','exchange.csv'])}
CHANNELS={n:len([c for c in pd.read_csv(p,nrows=2).columns if c.lower() not in {'date','timestamp'}]) for n,p in CSV.items()}
def data_args(n,batch_size,horizon):
    p=CSV[n]; data_name=n if n in {'ETTm1','ETTh2'} else 'custom'; freq={'ETTm1':'t','Weather':'10min','ETTh2':'h','Exchange':'d'}[n]
    return SimpleNamespace(task_name='long_term_forecast',data=data_name,root_path=str(p.parent)+'/',data_path=p.name,features='M',target='OT',freq=freq,seasonal_patterns='Monthly',seq_len=LOOKBACK,label_len=LABEL_LEN,pred_len=horizon,batch_size=batch_size,num_workers=0,embed='timeF',augmentation_ratio=0)
def build_data(horizon):
    out={}
    for n in DATASETS:
        full,_=data_provider(data_args(n,BATCH_SIZE,horizon),'train'); va,_=data_provider(data_args(n,EVAL_BATCH,horizon),'val')
        total=len(full); cal_len=max(MIN_CAL_WINDOWS,int(round(CAL_FRACTION*total))); gap=LOOKBACK+horizon
        fit_end=total-cal_len-gap; cal_start=fit_end+gap
        if fit_end<512 or cal_start>=total: raise RuntimeError(f'Insufficient split for {n}, H={horizon}: total={total}')
        fit=Subset(full,range(0,fit_end)); cal=Subset(full,range(cal_start,total))
        fit_loader=DataLoader(fit,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,drop_last=True,pin_memory=True)
        cal_loader=DataLoader(cal,batch_size=EVAL_BATCH,shuffle=False,num_workers=0,drop_last=False,pin_memory=True)
        val_loader=DataLoader(va,batch_size=EVAL_BATCH,shuffle=False,num_workers=0,drop_last=False,pin_memory=True)
        out[n]={'train_loader':fit_loader,'cal_loader':cal_loader,'val_loader':val_loader,'channels':CHANNELS[n],'fit_n':len(fit),'gap_n':gap,'cal_n':len(cal),'val_n':len(va)}
        print('H',horizon,n,'fit/gap/cal/official-val=',len(fit),gap,len(cal),len(va))
    return out


In [ ]:
# 3. Official model exposure and three controlled residual gates
def model_config(C):
    return SimpleNamespace(task_name='long_term_forecast',seq_len=LOOKBACK,pred_len=HORIZON,output_attention=False,patch_len=PATCH_LEN,stride=STRIDE,padding_patch='end',d_model=D_MODEL,n_heads=N_HEADS,e_layers=N_LAYERS,d_ff=D_FF,norm='BatchNorm',activation='gelu',dropout=DROPOUT,fc_dropout=DROPOUT,head_dropout=0.0,individual=False,enc_in=C,factor=1)
class OfficialIndependent(nn.Module):
    def __init__(self,C): super().__init__(); self.C=C; self.H=HORIZON; self.official=OfficialPatchTST(model_config(C))
    def direct(self,x):
        out=self.official(x,None,None,None); out=out[0] if isinstance(out,(tuple,list)) else out; return out
    def expose(self,x):
        means=x.mean(1,keepdim=True).detach(); xn=x-means; stdev=torch.sqrt(torch.var(xn,dim=1,keepdim=True,unbiased=False)+1e-5); xn=xn/stdev
        enc,nvars=self.official.patch_embedding(xn.permute(0,2,1)); enc,_=self.official.encoder(enc)
        enc=enc.reshape(-1,nvars,enc.shape[-2],enc.shape[-1]); head_in=enc.permute(0,1,3,2); yn=self.official.head(head_in); out=yn.permute(0,2,1)
        out=out*stdev[:,0,:].unsqueeze(1)+means[:,0,:].unsqueeze(1)
        return out,head_in.permute(0,1,3,2),stdev.permute(0,2,1)
    def forward(self,x): return self.direct(x)
def controller_features(x):
    z=x.permute(0,2,1); d=z[...,1:]-z[...,:-1]
    return torch.stack([z[...,-1],z.mean(-1),z.std(-1,unbiased=False),z[...,-1]-z[...,0],d.mean(-1),d.std(-1,unbiased=False),d.abs().mean(-1)],-1)
class SampleController(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Linear(7,32),nn.GELU(),nn.Linear(32,N_SLOTS)); nn.init.zeros_(self.net[-1].weight); nn.init.constant_(self.net[-1].bias,GATE_INIT_LOGIT)
    def forward(self,x): return torch.sigmoid(self.net(controller_features(x)))
class ShrinkController(nn.Module):
    def __init__(self,C):
        super().__init__(); self.mean_logit=nn.Parameter(torch.full((1,C,N_SLOTS),GATE_INIT_LOGIT))
        self.deviation=nn.Sequential(nn.Linear(7,32),nn.GELU(),nn.Linear(32,N_SLOTS))
        self.reliability=nn.Sequential(nn.Linear(7,16),nn.GELU(),nn.Linear(16,N_SLOTS))
        nn.init.zeros_(self.deviation[-1].weight); nn.init.zeros_(self.deviation[-1].bias)
        nn.init.zeros_(self.reliability[-1].weight); nn.init.constant_(self.reliability[-1].bias,-1.0)
    def components(self,x):
        f=controller_features(x); mean=torch.sigmoid(self.mean_logit).expand(len(x),-1,-1)
        deviation=MAX_GATE_DEVIATION*torch.tanh(self.deviation(f)); rho=torch.sigmoid(self.reliability(f))
        gate=(mean+rho*deviation).clamp(0.,1.); return gate,mean,deviation,rho
    def forward(self,x): return self.components(x)[0]
class SupportAdapter(nn.Module):
    def __init__(self,C,npatch,method):
        super().__init__(); self.method=method; self.attn=nn.MultiheadAttention(D_MODEL,N_HEADS,dropout=DROPOUT,batch_first=True); self.n1=nn.LayerNorm(D_MODEL); self.ff=nn.Sequential(nn.Linear(D_MODEL,D_FF),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(D_FF,D_MODEL)); self.n2=nn.LayerNorm(D_MODEL); self.head=nn.Sequential(nn.Flatten(-2),nn.Dropout(DROPOUT),nn.Linear(npatch*D_MODEL,HORIZON)); nn.init.zeros_(self.head[-1].weight); nn.init.zeros_(self.head[-1].bias)
        self.sample=SampleController(); self.mean_logit=nn.Parameter(torch.full((1,C,N_SLOTS),GATE_INIT_LOGIT)); self.shrink=ShrinkController(C)
    def delta(self,z):
        B,C,P,D=z.shape; q=z.permute(0,2,1,3).reshape(B*P,C,D); a,_=self.attn(q,q,q,need_weights=False); q=self.n1(q+a); q=self.n2(q+self.ff(q)); zc=q.reshape(B,P,C,D).permute(0,2,1,3); return MAX_DELTA*torch.tanh(self.head(zc-z))
    def gate(self,x):
        B,_,C=x.shape
        if self.method=='OfficialBoundedFixed': return torch.full((B,C,N_SLOTS),.5,device=x.device)
        if self.method=='OfficialMeanGate': return torch.sigmoid(self.mean_logit).expand(B,-1,-1)
        if self.method=='OfficialShrinkAdaptive': return self.shrink(x)
        return self.sample(x)
class BoundedModel(nn.Module):
    def __init__(self,base,method,npatch):
        super().__init__(); self.base=base; self.method=method; self.adapter=SupportAdapter(base.C,npatch,method)
        for p in self.base.parameters(): p.requires_grad=False
        if method!='OfficialMeanGate': self.adapter.mean_logit.requires_grad=False
        if method!='OfficialSampleAdaptive':
            for p in self.adapter.sample.parameters(): p.requires_grad=False
        if method!='OfficialShrinkAdaptive':
            for p in self.adapter.shrink.parameters(): p.requires_grad=False
    def train(self,mode=True): super().train(mode); self.base.eval(); return self
    def forward(self,x,gate_override=None,diagnostic=False):
        self.base.eval()
        with torch.no_grad(): base,z,sd=self.base.expose(x)
        dn=self.adapter.delta(z.detach()); g=self.adapter.gate(x) if gate_override is None else gate_override; gh=g.unsqueeze(-1).expand(-1,-1,-1,CHUNK).reshape(len(x),self.base.C,HORIZON); corr=gh*dn; pred=base+(corr*sd).permute(0,2,1)
        return (pred,base,g,dn,corr) if diagnostic else pred


In [ ]:
# 4. Preflight at all horizons: exact exposure, fallback, and disjoint split
for _h in HORIZONS:
    HORIZON=_h; N_SLOTS=HORIZON//CHUNK; DATA=build_data(HORIZON)
    seed_all(9801); C=CHANNELS['ETTm1']; base=OfficialIndependent(C).to(device); x=next(iter(DATA['ETTm1']['train_loader']))[0][:4].float().to(device); base.eval()
    with torch.no_grad(): y0=base.direct(x); y1,z,sd=base.expose(x)
    err=float((y0-y1).abs().max()); assert y0.shape==(4,HORIZON,C) and err<1e-5
    m=BoundedModel(copy.deepcopy(base),'OfficialShrinkAdaptive',z.shape[2]).to(device); m.eval()
    with torch.no_grad(): yp,yb,g,dn,c=m(x,torch.zeros_like(m.adapter.gate(x)),True)
    fallback=float((yp-yb).abs().max()); assert fallback==0.; print('H',HORIZON,'exposure',err,'fallback',fallback)
    del base,m,x,y0,y1,z,sd; gc.collect(); torch.cuda.empty_cache()
DATA={}


In [ ]:
# 5. Fit-prefix training; calibration tail is used for early stopping only
def unpack(batch):
    x,y=batch[0].float().to(device,non_blocking=True),batch[1].float().to(device,non_blocking=True); return x,y[:,-HORIZON:,:]
@torch.no_grad()
def evaluate_model(n,m,split='val',detail=False):
    m.eval(); sq=ab=cnt=0.; sample=[]
    for batch in DATA[n][split+'_loader']:
        x,y=unpack(batch)
        with torch.amp.autocast('cuda',dtype=torch.bfloat16,enabled=USE_BF16): pred=m(x)
        e=pred.float()-y; sq+=float(e.square().sum()); ab+=float(e.abs().sum()); cnt+=e.numel(); sample.append(e.square().mean((1,2)).cpu())
    out={'mse':sq/cnt,'mae':ab/cnt};
    if detail: out['sample_mse']=torch.cat(sample).numpy()
    return out
def train_base(n,seed):
    seed_all(seed); m=OfficialIndependent(CHANNELS[n]).to(device); p=CKPT_DIR/f'h{HORIZON}'/'base'/n/f'{seed}.pth'; p.parent.mkdir(parents=True,exist_ok=True)
    if p.exists() and not FORCE_RETRAIN: st=torch.load(str(p),map_location='cpu',weights_only=False); m.load_state_dict(st['model']); print('[LOAD]',p); return m.cpu(),st
    opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=WD); sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=LR,epochs=TRAIN_EPOCHS,steps_per_epoch=len(DATA[n]['train_loader']),pct_start=.2,anneal_strategy='cos'); best=1e99; bad=0; bst=None; hist=[]
    for ep in range(TRAIN_EPOCHS):
        m.train(); losses=[]
        for batch in DATA[n]['train_loader']:
            x,y=unpack(batch); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',dtype=torch.bfloat16,enabled=USE_BF16): loss=F.mse_loss(m(x).float(),y)
            loss.backward(); nn.utils.clip_grad_norm_(m.parameters(),1.); opt.step(); sched.step(); losses.append(float(loss))
        v=evaluate_model(n,m,'cal')['mse']; row={'epoch':ep+1,'cal_mse':v,'train_loss':float(np.mean(losses))}; hist.append(row); print('H',HORIZON,n,'base',seed,row)
        if v<best-1e-7: best=v; bad=0; bst=copy.deepcopy(m.state_dict())
        else: bad+=1
        if bad>=PATIENCE: break
    m.load_state_dict(bst); st={'model':bst,'best_cal':best,'history':hist,'test_evaluated':False}; torch.save(st,str(p)); return m.cpu(),st
def train_support(n,seed,base,npatch):
    seed_all(seed); m=BoundedModel(copy.deepcopy(base),'OfficialShrinkAdaptive',npatch).to(device); p=CKPT_DIR/f'h{HORIZON}'/'support'/n/f'{seed}.pth'; p.parent.mkdir(parents=True,exist_ok=True)
    if p.exists() and not FORCE_RETRAIN: st=torch.load(str(p),map_location='cpu',weights_only=False); m.adapter.load_state_dict(st['adapter']); print('[LOAD]',p); return m.cpu(),st
    pars=[q for q in m.adapter.parameters() if q.requires_grad]; opt=torch.optim.AdamW(pars,lr=ADAPTER_LR,weight_decay=WD); sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=ADAPTER_LR,epochs=ADAPTER_EPOCHS,steps_per_epoch=len(DATA[n]['train_loader']),pct_start=.2,anneal_strategy='cos'); best=1e99; bad=0; bst=None; hist=[]
    for ep in range(ADAPTER_EPOCHS):
        m.train(); losses=[]
        for batch in DATA[n]['train_loader']:
            x,y=unpack(batch); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',dtype=torch.bfloat16,enabled=USE_BF16):
                pred,_,g,_,_=m(x,diagnostic=True); _,anchor,_,_=m.adapter.shrink.components(x); loss=F.mse_loss(pred.float(),y)+SHRINK_LAMBDA*(g-anchor).square().mean()
            loss.backward(); nn.utils.clip_grad_norm_(pars,1.); opt.step(); sched.step(); losses.append(float(loss))
        v=evaluate_model(n,m,'cal')['mse']; row={'epoch':ep+1,'cal_mse':v,'train_loss':float(np.mean(losses))}; hist.append(row); print('H',HORIZON,n,'support',seed,row)
        if v<best-1e-7: best=v; bad=0; bst=copy.deepcopy(m.adapter.state_dict())
        else: bad+=1
        if bad>=PATIENCE: break
    m.adapter.load_state_dict(bst); st={'adapter':bst,'best_cal':best,'history':hist}; torch.save(st,str(p)); return m.cpu(),st


In [ ]:
# 6. Calibration estimators and official-validation evaluation
@torch.no_grad()
def collect_components(n,m,split):
    m.eval(); bases=[]; deltas=[]; ys=[]
    for batch in DATA[n][split+'_loader']:
        x,y=unpack(batch)
        with torch.amp.autocast('cuda',dtype=torch.bfloat16,enabled=USE_BF16): pred,base,g,dn,corr=m(x,diagnostic=True)
        bases.append(base.float().cpu()); deltas.append((pred-base).float().cpu()); ys.append(y.float().cpu())
    return torch.cat(bases),torch.cat(deltas),torch.cat(ys)
def analytic_alpha(base,delta,y,dims):
    e=base-y; num=-(e*delta).sum(dim=dims); den=delta.square().sum(dim=dims).clamp_min(1e-12); return (num/den).clamp(0.,1.)
def mse_alpha(base,delta,y,alpha): return float((base+delta*alpha-y).square().mean())
def calibration_policy(base,delta,y):
    # Global closed-form alpha.
    ag=analytic_alpha(base,delta,y,(0,1,2))
    # Cross-fit safety: estimate on one temporal half and retain it only if it
    # improves the opposite half. This can exactly fall back to alpha=0.
    cut=len(base)//2; accepted=[]
    for fit_idx,check_idx in [(slice(0,cut),slice(cut,None)),(slice(cut,None),slice(0,cut))]:
        a=analytic_alpha(base[fit_idx],delta[fit_idx],y[fit_idx],(0,1,2))
        if mse_alpha(base[check_idx],delta[check_idx],y[check_idx],a)<mse_alpha(base[check_idx],delta[check_idx],y[check_idx],0.): accepted.append(float(a))
    safe=torch.tensor(float(np.mean(accepted)) if accepted else 0.)
    # Per-channel alpha shrunk toward the global estimate according to correction energy.
    rawc=analytic_alpha(base,delta,y,(0,1)); energy=delta.square().sum((0,1)); prior=torch.median(energy).clamp_min(1e-12); w=energy/(energy+prior); ac=(w*rawc+(1-w)*ag).view(1,1,-1)
    return {'CalGlobal':ag,'CrossFitSafe':safe,'CalChannelShrink':ac}, {'global_alpha':float(ag),'safe_alpha':float(safe),'accepted_halves':len(accepted),'channel_alpha_mean':float(ac.mean()),'channel_alpha_min':float(ac.min()),'channel_alpha_max':float(ac.max())}

ROWS=[]; DETAIL={}; CAL=[]; SPLITS=[]
for _h in HORIZONS:
    HORIZON=_h; N_SLOTS=HORIZON//CHUNK; DATA=build_data(HORIZON)
    for n in DATASETS:
        SPLITS.append({'horizon':HORIZON,'dataset':n,**{k:DATA[n][k] for k in ['fit_n','gap_n','cal_n','val_n']}})
        for seed in SEEDS:
            base,bst=train_base(n,seed); base=base.to(device); bx=next(iter(DATA[n]['train_loader']))[0][:2].float().to(device); base.eval()
            with torch.no_grad(): _,z,_=base.expose(bx)
            m,st=train_support(n,seed,base.cpu(),z.shape[2]); m=m.to(device)
            cb,cd,cy=collect_components(n,m,'cal'); vb,vd,vy=collect_components(n,m,'val'); policies,diag=calibration_policy(cb,cd,cy)
            policies['Independent']=torch.tensor(0.); policies['NaturalSupport']=torch.tensor(1.); policies['ValOracle']=analytic_alpha(vb,vd,vy,(0,1,2))
            CAL.append({'horizon':HORIZON,'dataset':n,'seed':seed,'oracle_alpha':float(policies['ValOracle']),**diag})
            for method,a in policies.items():
                pred=vb+vd*a; e=pred-vy; sm=e.square().mean((1,2)).numpy(); mse=float(e.square().mean()); mae=float(e.abs().mean())
                ROWS.append({'horizon':HORIZON,'dataset':n,'seed':seed,'method':method,'mse':mse,'mae':mae,'alpha_mean':float(torch.as_tensor(a).mean())}); DETAIL[(HORIZON,n,method,seed)]=sm
            pd.DataFrame(ROWS).to_csv(RESULT_DIR/'validation_by_seed.partial.csv',index=False); pd.DataFrame(CAL).to_csv(RESULT_DIR/'calibration.partial.csv',index=False)
            del base,m,bx,z,cb,cd,cy,vb,vd,vy; gc.collect(); torch.cuda.empty_cache()
    DATA={}; gc.collect(); torch.cuda.empty_cache()
result_df=pd.DataFrame(ROWS); cal_df=pd.DataFrame(CAL); split_df=pd.DataFrame(SPLITS)
result_df.to_csv(RESULT_DIR/'validation_by_seed.csv',index=False); cal_df.to_csv(RESULT_DIR/'calibration.csv',index=False); split_df.to_csv(RESULT_DIR/'temporal_splits.csv',index=False)
summary=result_df.groupby(['horizon','dataset','method']).agg(mse=('mse','mean'),std=('mse','std'),mae=('mae','mean'),alpha=('alpha_mean','mean'),seeds=('seed','nunique')).reset_index(); summary.to_csv(RESULT_DIR/'summary.csv',index=False); display(summary.sort_values(['horizon','dataset','mse']))


In [ ]:
# 7. Paired block bootstrap and decision
def mb(a,b,seed,block):
    d=np.asarray(b)-np.asarray(a); rng=np.random.default_rng(seed); vals=[]; n=len(d); block=min(block,n)
    for _ in range(N_BOOT):
        starts=rng.integers(0,max(1,n-block+1),size=int(np.ceil(n/block))); vals.append(np.concatenate([d[s:s+block] for s in starts])[:n].mean())
    return float(d.mean()),float(np.quantile(vals,.025)),float(np.quantile(vals,.975))
B=[]
for h in HORIZONS:
 for n in DATASETS:
  for seed in SEEDS:
   for cand,base in [('NaturalSupport','Independent'),('CalGlobal','Independent'),('CrossFitSafe','Independent'),('CalChannelShrink','Independent'),('CrossFitSafe','NaturalSupport'),('CalChannelShrink','NaturalSupport'),('ValOracle','Independent')]:
    d,lo,hi=mb(DETAIL[(h,n,base,seed)],DETAIL[(h,n,cand,seed)],seed+h,h); B.append({'horizon':h,'dataset':n,'comparison':cand+' - '+base,'seed':seed,'delta':d,'ci_low':lo,'ci_high':hi})
boot=pd.DataFrame(B); boot.to_csv(RESULT_DIR/'paired_bootstrap_by_seed.csv',index=False)
boot_summary=boot.groupby(['horizon','dataset','comparison']).agg(mean_delta=('delta','mean'),positive_seeds=('delta',lambda x:int((x>0).sum())),negative_seeds=('delta',lambda x:int((x<0).sum())),min_ci_low=('ci_low','min'),max_ci_high=('ci_high','max')).reset_index(); boot_summary.to_csv(RESULT_DIR/'paired_bootstrap_summary.csv',index=False)
wide=result_df.pivot(index=['horizon','dataset','seed'],columns='method',values='mse').reset_index()
for m in ['NaturalSupport','CalGlobal','CrossFitSafe','CalChannelShrink','ValOracle']: wide[m+'_gain']=(wide.Independent-wide[m])/wide.Independent
def hierarchical_ci(col,seed):
    rng=np.random.default_rng(seed); strata=[(h,n) for h in HORIZONS for n in DATASETS]; vals=[]
    for _ in range(10000):
        chosen=[strata[i] for i in rng.integers(0,len(strata),len(strata))]; per=[]
        for h,n in chosen:
            a=wide.query('horizon==@h and dataset==@n')[col].to_numpy(); per.append(float(rng.choice(a,size=len(a),replace=True).mean()))
        vals.append(float(np.mean(per)))
    obs=float(wide.groupby(['horizon','dataset'])[col].mean().mean()); return {'metric':col,'mean':obs,'ci_low':float(np.quantile(vals,.025)),'ci_high':float(np.quantile(vals,.975))}
hier=pd.DataFrame([hierarchical_ci(m+'_gain',981000+i) for i,m in enumerate(['NaturalSupport','CalGlobal','CrossFitSafe','CalChannelShrink','ValOracle'])]); hier.to_csv(RESULT_DIR/'hierarchical_bootstrap.csv',index=False)
agg=summary.pivot(index=['horizon','dataset'],columns='method',values='mse').reset_index(); agg['best_deployable']=agg[['Independent','NaturalSupport','CalGlobal','CrossFitSafe','CalChannelShrink']].idxmin(axis=1)
agg['safe_gain']=(agg.Independent-agg.CrossFitSafe)/agg.Independent; agg['safe_regret_vs_oracle']=(agg.CrossFitSafe-agg.ValOracle)/agg.ValOracle
row=hier.query("metric=='CrossFitSafe_gain'").iloc[0]; improve=int((agg.safe_gain>0).sum()); exchange720=agg.query("horizon==720 and dataset=='Exchange'").iloc[0]
detects_failure=bool(exchange720.CrossFitSafe <= 1.005*exchange720.Independent); worst_reg=float(((agg.CrossFitSafe-agg.Independent)/agg.Independent).max())
passed=bool(improve>=10 and row.ci_low>0 and detects_failure and worst_reg<=.005)
if passed: conclusion='calibration_safe_support_validated'; next_step='freeze_model_and_run_test_once'
elif detects_failure and row['mean']>0: conclusion='safety_gate_promising_but_not_strict'; next_step='replicate_with_five_seeds_or_add_datasets'
else: conclusion='calibration_safety_not_supported'; next_step='use_horizon_policy_or_independent_baseline'
decision={'experiment':EXP_NAME,'test_evaluated':False,'completed_rows':len(result_df),'expected_rows':len(HORIZONS)*len(DATASETS)*len(SEEDS)*len(METHODS),'safe_beats_independent_cells':improve,'total_cells':len(agg),'overall_safe_gain':float(row['mean']),'overall_95ci':[float(row.ci_low),float(row.ci_high)],'exchange_h720_failure_detected':detects_failure,'exchange_h720_safe_alpha_mean':float(cal_df.query("horizon==720 and dataset=='Exchange'").safe_alpha.mean()),'worst_regression_vs_independent':worst_reg,'strict_pass':passed,'conclusion':conclusion,'next':next_step}
agg.to_csv(RESULT_DIR/'comparison.csv',index=False); wide.to_csv(RESULT_DIR/'relative_gain_by_seed.csv',index=False); (RESULT_DIR/'decision.json').write_text(json.dumps(decision,indent=2))
print('='*100); print('EXPERIMENT 98 — FINAL RESULT'); print('='*100); print('Test set evaluated: NO'); print('Official validation used only after calibration: YES'); print('\n[Aggregate]'); display(summary.sort_values(['horizon','dataset','mse'])); print('\n[Comparison]'); display(agg); print('\n[Calibration]'); display(cal_df); print('\n[Hierarchical bootstrap]'); display(hier); print('\n[Decision]'); print(json.dumps(decision,indent=2))
